In [2]:
# ============================================================
# CELL 1 — Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import os
import json
import time
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

In [4]:
# ============================================================
# CELL 3 — Paths
# ============================================================
RAW_DIR = '/content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all250'
PROCESSED_DIR = '/content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all250'

os.makedirs(PROCESSED_DIR, exist_ok=True)

In [5]:
# ============================================================
# CELL 4 — Reproducibility
# ============================================================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [6]:
# ============================================================
# CELL 5 — Device
# ============================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Device:", device)

if device.type == 'cuda':
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

Device: cuda
GPU: Tesla T4
VRAM: 15.637086208 GB


In [7]:
# ============================================================
# CELL 6 — Helper Loader
# ============================================================
def load_from_processed_or_raw(fname):
    p1 = os.path.join(PROCESSED_DIR, fname)
    p2 = os.path.join(RAW_DIR, fname)

    if os.path.exists(p1):
        return p1
    elif os.path.exists(p2):
        return p2
    else:
        raise FileNotFoundError(f"Could not find {fname}")

In [8]:
# ============================================================
# CELL 7 — Load Full 250-Sign Dataset
# ============================================================
X_train = np.load(os.path.join(PROCESSED_DIR, 'X_train_norm.npz'))['data']
X_val   = np.load(os.path.join(PROCESSED_DIR, 'X_val_norm.npz'))['data']
X_test  = np.load(os.path.join(PROCESSED_DIR, 'X_test_norm.npz'))['data']

y_train = np.load(load_from_processed_or_raw('y_train.npy'))
y_val   = np.load(load_from_processed_or_raw('y_val.npy'))
y_test  = np.load(load_from_processed_or_raw('y_test.npy'))

lengths_train = np.load(load_from_processed_or_raw('lengths_train.npy'))
lengths_val   = np.load(load_from_processed_or_raw('lengths_val.npy'))
lengths_test  = np.load(load_from_processed_or_raw('lengths_test.npy'))

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:  ", y_val.shape)
print("y_test: ", y_test.shape)

X_train: (55642, 96, 708)
X_val:   (9291, 96, 708)
X_test:  (11826, 96, 708)
y_train: (55642,)
y_val:   (9291,)
y_test:  (11826,)


In [9]:
# ============================================================
# CELL 8 — Load Metadata
# ============================================================
with open(load_from_processed_or_raw('metadata.json')) as f:
    metadata = json.load(f)

if 'label_to_word' in metadata:
    idx_to_sign = {int(k): v for k, v in metadata['label_to_word'].items()}
    sign_to_idx = {v: k for k, v in idx_to_sign.items()}
elif 'labels' in metadata:
    idx_to_sign = {int(k): v for k, v in metadata['labels'].items()}
    sign_to_idx = {v: k for k, v in idx_to_sign.items()}
elif 'signs' in metadata:
    idx_to_sign = {int(k): v for k, v in enumerate(metadata['signs'])}
    sign_to_idx = {v: k for k, v in idx_to_sign.items()}
else:
    raise KeyError("Could not find label mapping in metadata.")

N_CLASSES = len(idx_to_sign)

print("N_CLASSES:", N_CLASSES)
print("First 10 signs:")
for i in range(10):
    print(i, idx_to_sign[i])

N_CLASSES: 250
First 10 signs:
0 TV
1 after
2 airplane
3 all
4 alligator
5 animal
6 another
7 any
8 apple
9 arm


In [10]:
# ============================================================
# CELL 9 — Save Full Label Map
# ============================================================
full_label_map = {
    'n_classes': N_CLASSES,
    'idx_to_sign': {str(k): v for k, v in idx_to_sign.items()},
    'sign_to_idx': {k: int(v) for k, v in sign_to_idx.items()},
}

label_map_path = os.path.join(PROCESSED_DIR, 'full250_label_map.json')

with open(label_map_path, 'w') as f:
    json.dump(full_label_map, f, indent=2)

print("Saved:", label_map_path)

Saved: /content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all250/full250_label_map.json


In [11]:
# ============================================================
# CELL 10 — Compute Full Class Weights
# ============================================================
classes = np.arange(N_CLASSES)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
).astype(np.float32)

np.save(os.path.join(PROCESSED_DIR, 'class_weights_250.npy'), class_weights)

print("Class weights shape:", class_weights.shape)
print("Min:", class_weights.min())
print("Max:", class_weights.max())

Class weights shape: (250,)
Min: 0.7920569
Max: 1.752504


In [12]:
# ============================================================
# CELL 11 — Dataset Class
# ============================================================
class SignDataset(Dataset):
    def __init__(self, X, y, lengths, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.lengths = torch.tensor(lengths, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        l = self.lengths[idx]

        if self.augment:
            x, l = self._augment(x, l)

        return x, y, l

    def _augment(self, x, l):
        x = x.clone()
        l_int = int(l.item())

        # Random time stretch
        if torch.rand(1).item() < 0.5:
            rate = 0.85 + torch.rand(1).item() * 0.30
            new_len = max(1, int(round(l_int * rate)))

            real = x[:l_int]
            old_idx = torch.linspace(0, l_int - 1, new_len)

            lo = old_idx.floor().long().clamp(0, l_int - 1)
            hi = old_idx.ceil().long().clamp(0, l_int - 1)
            frac = (old_idx - lo.float()).unsqueeze(1)

            stretched = real[lo] * (1 - frac) + real[hi] * frac

            out = torch.zeros_like(x)
            keep = min(new_len, x.shape[0])
            out[:keep] = stretched[:keep]

            x = out
            l_int = keep

        # Small noise
        if torch.rand(1).item() < 0.5:
            x[:l_int] += torch.randn(l_int, x.shape[1]) * 0.001

        # Frame dropout
        if torch.rand(1).item() < 0.3:
            mask = torch.rand(l_int) < 0.05
            x[:l_int][mask] = 0.0

        return x, torch.tensor(l_int, dtype=torch.long)

In [13]:
# ============================================================
# CELL 12 — DataLoaders
# ============================================================
BATCH = 128

train_dataset = SignDataset(X_train, y_train, lengths_train, augment=True)
val_dataset   = SignDataset(X_val, y_val, lengths_val, augment=False)
test_dataset  = SignDataset(X_test, y_test, lengths_test, augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

x_batch, y_batch, l_batch = next(iter(train_loader))

print("Batch X:", x_batch.shape)
print("Batch y:", y_batch.shape)
print("Batch lengths:", l_batch.shape)

Batch X: torch.Size([128, 96, 708])
Batch y: torch.Size([128])
Batch lengths: torch.Size([128])


In [14]:




# ============================================================
# CELL 13 — Shared Model Blocks
# ============================================================
class ConvBlock1D(nn.Module):
    def __init__(self, channels, kernel_size=5, dilation=1, dropout=0.25):
        super().__init__()

        padding = dilation * (kernel_size - 1) // 2

        self.net = nn.Sequential(
            nn.Conv1d(
                channels,
                channels,
                kernel_size=kernel_size,
                padding=padding,
                dilation=dilation,
                groups=1,
                bias=False
            ),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Conv1d(
                channels,
                channels,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.net(x)


class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, channels, kernel_size=5, dilation=1, dropout=0.25):
        super().__init__()

        padding = dilation * (kernel_size - 1) // 2

        self.depthwise = nn.Conv1d(
            channels,
            channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
            groups=channels,
            bias=False
        )

        self.pointwise = nn.Conv1d(
            channels,
            channels,
            kernel_size=1,
            bias=False
        )

        self.net = nn.Sequential(
            self.depthwise,
            self.pointwise,
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return x + self.net(x)


def masked_mean_pool_time(x, lengths):
    """
    x shape for sequence models: (B, T, C)
    """
    B, T, C = x.shape

    mask = torch.arange(T, device=x.device).unsqueeze(0) < lengths.unsqueeze(1)
    mask = mask.unsqueeze(-1).float()

    return (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)


def masked_mean_pool_conv(x, lengths):
    """
    x shape for conv models: (B, C, T)
    """
    B, C, T = x.shape

    mask = torch.arange(T, device=x.device).unsqueeze(0) < lengths.unsqueeze(1)
    mask = mask.unsqueeze(1).float()

    return (x * mask).sum(dim=2) / mask.sum(dim=2).clamp(min=1)

In [15]:
# ============================================================
# CELL 14 — Strong CNN
# ============================================================
class StrongSignCNN(nn.Module):
    def __init__(
        self,
        input_size=708,
        channels=384,
        n_classes=250,
        dropout=0.25
    ):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_size, channels),
            nn.LayerNorm(channels),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.blocks = nn.Sequential(
            DepthwiseSeparableBlock(channels, kernel_size=3, dilation=1, dropout=dropout),
            DepthwiseSeparableBlock(channels, kernel_size=5, dilation=1, dropout=dropout),
            DepthwiseSeparableBlock(channels, kernel_size=5, dilation=2, dropout=dropout),
            DepthwiseSeparableBlock(channels, kernel_size=7, dilation=2, dropout=dropout),
            DepthwiseSeparableBlock(channels, kernel_size=7, dilation=4, dropout=dropout),
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(channels),
            nn.Linear(channels, channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(channels, n_classes)
        )

    def forward(self, x, lengths=None):
        x = self.input_proj(x)
        x = x.transpose(1, 2)

        x = self.blocks(x)

        if lengths is not None:
            x = masked_mean_pool_conv(x, lengths)
        else:
            x = x.mean(dim=2)

        return self.classifier(x)

In [16]:
# ============================================================
# CELL 15 — Strong CNN + GRU
# ============================================================
class StrongSignCNNGRU(nn.Module):
    def __init__(
        self,
        input_size=708,
        channels=384,
        hidden=384,
        n_classes=250,
        dropout=0.25
    ):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_size, channels),
            nn.LayerNorm(channels),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.conv_blocks = nn.Sequential(
            DepthwiseSeparableBlock(channels, kernel_size=3, dilation=1, dropout=dropout),
            DepthwiseSeparableBlock(channels, kernel_size=5, dilation=1, dropout=dropout),
            DepthwiseSeparableBlock(channels, kernel_size=5, dilation=2, dropout=dropout),
        )

        self.gru = nn.GRU(
            input_size=channels,
            hidden_size=hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes)
        )

    def forward(self, x, lengths=None):
        x = self.input_proj(x)
        x = x.transpose(1, 2)
        x = self.conv_blocks(x)
        x = x.transpose(1, 2)

        out, _ = self.gru(x)

        if lengths is not None:
            idx = (lengths - 1).clamp(min=0)
            out = out[torch.arange(len(idx), device=x.device), idx]
        else:
            out = out[:, -1]

        return self.classifier(out)

In [17]:
# ============================================================
# CELL 16 — Strong Transformer
# ============================================================
class StrongSignTransformer(nn.Module):
    def __init__(
        self,
        input_size=708,
        d_model=384,
        nhead=6,
        layers=4,
        n_classes=250,
        dropout=0.25
    ):
        super().__init__()

        self.proj = nn.Sequential(
            nn.Linear(input_size, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.pos_emb = nn.Embedding(96, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=1024,
            dropout=dropout,
            batch_first=True,
            activation='gelu',
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes)
        )

    def forward(self, x, lengths=None):
        B, T, _ = x.shape

        positions = torch.arange(T, device=x.device)
        x = self.proj(x) + self.pos_emb(positions)

        if lengths is not None:
            mask = torch.arange(T, device=x.device).unsqueeze(0) >= lengths.unsqueeze(1)
        else:
            mask = None

        out = self.encoder(x, src_key_padding_mask=mask)

        if lengths is not None:
            out = masked_mean_pool_time(out, lengths)
        else:
            out = out.mean(dim=1)

        return self.classifier(out)

In [18]:
# ============================================================
# CELL 17 — Strong TCN
# ============================================================
class StrongSignTCN(nn.Module):
    def __init__(
        self,
        input_size=708,
        d_model=384,
        n_classes=250,
        dropout=0.25
    ):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_size, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.tcn_blocks = nn.Sequential(
            DepthwiseSeparableBlock(d_model, kernel_size=3, dilation=1, dropout=dropout),
            DepthwiseSeparableBlock(d_model, kernel_size=5, dilation=2, dropout=dropout),
            DepthwiseSeparableBlock(d_model, kernel_size=5, dilation=4, dropout=dropout),
            DepthwiseSeparableBlock(d_model, kernel_size=7, dilation=8, dropout=dropout),
            DepthwiseSeparableBlock(d_model, kernel_size=7, dilation=16, dropout=dropout),
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes)
        )

    def forward(self, x, lengths=None):
        x = self.input_proj(x)

        # TCN expects shape: [batch, channels, time]
        x = x.transpose(1, 2)
        x = self.tcn_blocks(x)
        x = x.transpose(1, 2)

        if lengths is not None:
            out = masked_mean_pool_time(x, lengths)
        else:
            out = x.mean(dim=1)

        return self.classifier(out)

In [19]:
# ============================================================
# CELL 18 — Model Registry
# ============================================================
model_builders = {
    'StrongCNN': StrongSignCNN,
    'StrongCNN_GRU': StrongSignCNNGRU,
    'StrongTransformer': StrongSignTransformer,
    'StrongTCN': StrongSignTCN,
}

print("Models:")
for name in model_builders:
    print("-", name)

Models:
- StrongCNN
- StrongCNN_GRU
- StrongTransformer
- StrongTCN


In [20]:
# ============================================================
# CELL 19 — Dummy Forward Pass
# ============================================================
dummy_x = torch.randn(4, 96, 708).to(device)
dummy_l = torch.randint(20, 96, (4,)).to(device)

for name, builder in model_builders.items():
    model = builder(n_classes=N_CLASSES).to(device)

    out = model(dummy_x, dummy_l)
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"{name:24s} output={str(out.shape):15s} params={params:,}")

    del model
    torch.cuda.empty_cache()

StrongCNN                output=torch.Size([4, 250]) params=1,269,370
StrongCNN_GRU            output=torch.Size([4, 250]) params=5,548,666


/tmp/ipykernel_13064/3882962478.py:35: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)


StrongTransformer        output=torch.Size([4, 250]) params=6,077,690
StrongTCN                output=torch.Size([4, 250]) params=1,269,370


In [21]:
# ============================================================
# CELL 20 — Accuracy Helpers
# ============================================================
def topk_acc(logits, targets, k):
    _, pred = logits.topk(k, dim=1)
    correct = pred.eq(targets.view(-1, 1).expand_as(pred))
    return correct.any(dim=1).float().mean().item()

In [22]:
# ============================================================
# CELL 21 — Train One Epoch
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    top1 = 0.0
    top5 = 0.0
    top10 = 0.0

    for x, y, lengths in loader:
        x = x.to(device)
        y = y.to(device)
        lengths = lengths.to(device)

        optimizer.zero_grad()

        logits = model(x, lengths)
        loss = criterion(logits, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        top1 += topk_acc(logits, y, 1)
        top5 += topk_acc(logits, y, 5)
        top10 += topk_acc(logits, y, 10)

    n = len(loader)

    return total_loss / n, top1 / n, top5 / n, top10 / n

In [23]:
# ============================================================
# CELL 22 — Evaluation
# ============================================================
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    top1 = 0.0
    top5 = 0.0
    top10 = 0.0

    for x, y, lengths in loader:
        x = x.to(device)
        y = y.to(device)
        lengths = lengths.to(device)

        logits = model(x, lengths)
        loss = criterion(logits, y)

        total_loss += loss.item()
        top1 += topk_acc(logits, y, 1)
        top5 += topk_acc(logits, y, 5)
        top10 += topk_acc(logits, y, 10)

    n = len(loader)

    return total_loss / n, top1 / n, top5 / n, top10 / n

In [24]:
# ============================================================
# CELL 23 — Train Full Model Function
# ============================================================
def train_model(model, model_name, epochs=50, lr=2e-4):
    model = model.to(device)

    weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights_tensor, label_smoothing=0.05)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs
    )

    best_val_top1 = 0.0
    best_ckpt_path = os.path.join(PROCESSED_DIR, f'best_full250_{model_name}.pt')

    history = []
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        tr_loss, tr_top1, tr_top5, tr_top10 = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )

        vl_loss, vl_top1, vl_top5, vl_top10 = evaluate(
            model, val_loader, criterion, device
        )

        scheduler.step()

        row = {
            'epoch': epoch,
            'train_loss': tr_loss,
            'train_top1': tr_top1,
            'train_top5': tr_top5,
            'train_top10': tr_top10,
            'val_loss': vl_loss,
            'val_top1': vl_top1,
            'val_top5': vl_top5,
            'val_top10': vl_top10,
            'lr': scheduler.get_last_lr()[0],
        }

        history.append(row)

        if vl_top1 > best_val_top1:
            best_val_top1 = vl_top1
            torch.save(model.state_dict(), best_ckpt_path)

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"[{model_name}] Epoch {epoch:03d}/{epochs} | "
                f"tr_loss={tr_loss:.4f} tr_top1={tr_top1:.3f} tr_top5={tr_top5:.3f} | "
                f"vl_loss={vl_loss:.4f} vl_top1={vl_top1:.3f} vl_top5={vl_top5:.3f}"
            )

    elapsed_min = (time.time() - start_time) / 60

    hist_df = pd.DataFrame(history)
    hist_path = os.path.join(PROCESSED_DIR, f'history_full250_{model_name}.csv')
    hist_df.to_csv(hist_path, index=False)

    model.load_state_dict(torch.load(best_ckpt_path, map_location=device))

    te_loss, te_top1, te_top5, te_top10 = evaluate(
        model, test_loader, criterion, device
    )

    return {
        'model': model_name,
        'best_val_top1': best_val_top1,
        'test_loss': te_loss,
        'test_top1': te_top1,
        'test_top5': te_top5,
        'test_top10': te_top10,
        'train_time_min': elapsed_min,
        'params': sum(p.numel() for p in model.parameters() if p.requires_grad),
        'ckpt_path': best_ckpt_path,
        'history_path': hist_path,
    }

In [25]:
# ============================================================
# CELL 24 — Choose Full-Dataset Models
# ============================================================
MODELS_TO_TRAIN = [
    'StrongCNN',
    'StrongCNN_GRU',
    'StrongTransformer',
    'StrongTCN',
]

EPOCHS = 100

print("Training:")
for m in MODELS_TO_TRAIN:
    print("-", m)

Training:
- StrongCNN
- StrongCNN_GRU
- StrongTransformer
- StrongTCN


In [26]:
# ============================================================
# CELL 25 — Train All Four Full Models
# ============================================================
results = {}

for model_name in MODELS_TO_TRAIN:
    print("\n" + "=" * 80)
    print(f"Training {model_name}")
    print("=" * 80)

    model = model_builders[model_name](n_classes=N_CLASSES)

    results[model_name] = train_model(
        model=model,
        model_name=model_name,
        epochs=EPOCHS,
        lr=2e-4
    )

    print("Done:")
    print(results[model_name])

    del model
    torch.cuda.empty_cache()


Training StrongCNN
[StrongCNN] Epoch 001/100 | tr_loss=4.8699 tr_top1=0.053 tr_top5=0.181 | vl_loss=4.3007 vl_top1=0.127 vl_top5=0.361
[StrongCNN] Epoch 005/100 | tr_loss=2.7161 tr_top1=0.428 tr_top5=0.742 | vl_loss=3.0157 vl_top1=0.377 vl_top5=0.670
[StrongCNN] Epoch 010/100 | tr_loss=1.9360 tr_top1=0.625 tr_top5=0.874 | vl_loss=2.7292 vl_top1=0.455 vl_top5=0.720
[StrongCNN] Epoch 015/100 | tr_loss=1.5748 tr_top1=0.722 tr_top5=0.923 | vl_loss=2.6537 vl_top1=0.477 vl_top5=0.734
[StrongCNN] Epoch 020/100 | tr_loss=1.3523 tr_top1=0.782 tr_top5=0.949 | vl_loss=2.5220 vl_top1=0.532 vl_top5=0.765
[StrongCNN] Epoch 025/100 | tr_loss=1.1949 tr_top1=0.828 tr_top5=0.968 | vl_loss=2.4713 vl_top1=0.547 vl_top5=0.770
[StrongCNN] Epoch 030/100 | tr_loss=1.0710 tr_top1=0.864 tr_top5=0.980 | vl_loss=2.5088 vl_top1=0.546 vl_top5=0.764
[StrongCNN] Epoch 035/100 | tr_loss=0.9772 tr_top1=0.895 tr_top5=0.989 | vl_loss=2.5123 vl_top1=0.566 vl_top5=0.771
[StrongCNN] Epoch 040/100 | tr_loss=0.9014 tr_top1=0

/tmp/ipykernel_13064/3882962478.py:35: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)


[StrongTransformer] Epoch 001/100 | tr_loss=4.9118 tr_top1=0.046 tr_top5=0.162 | vl_loss=4.3850 vl_top1=0.095 vl_top5=0.313
[StrongTransformer] Epoch 005/100 | tr_loss=2.7488 tr_top1=0.414 tr_top5=0.733 | vl_loss=3.0510 vl_top1=0.389 vl_top5=0.684
[StrongTransformer] Epoch 010/100 | tr_loss=1.8716 tr_top1=0.636 tr_top5=0.881 | vl_loss=2.6415 vl_top1=0.491 vl_top5=0.748
[StrongTransformer] Epoch 015/100 | tr_loss=1.4571 tr_top1=0.744 tr_top5=0.933 | vl_loss=2.5684 vl_top1=0.511 vl_top5=0.755
[StrongTransformer] Epoch 020/100 | tr_loss=1.2017 tr_top1=0.818 tr_top5=0.962 | vl_loss=2.4131 vl_top1=0.559 vl_top5=0.777
[StrongTransformer] Epoch 025/100 | tr_loss=1.0127 tr_top1=0.870 tr_top5=0.981 | vl_loss=2.5323 vl_top1=0.561 vl_top5=0.772
[StrongTransformer] Epoch 030/100 | tr_loss=0.8847 tr_top1=0.912 tr_top5=0.991 | vl_loss=2.5352 vl_top1=0.563 vl_top5=0.770
[StrongTransformer] Epoch 035/100 | tr_loss=0.7951 tr_top1=0.940 tr_top5=0.996 | vl_loss=2.6497 vl_top1=0.546 vl_top5=0.766
[StrongT

In [27]:
# ============================================================
# CELL 26 — Ablation Table
# ============================================================
rows = []

for name, r in results.items():
    rows.append({
        'Model': r['model'],
        'Best Val Top-1 (%)': round(r['best_val_top1'] * 100, 2),
        'Test Top-1 (%)': round(r['test_top1'] * 100, 2),
        'Test Top-5 (%)': round(r['test_top5'] * 100, 2),
        'Test Top-10 (%)': round(r['test_top10'] * 100, 2),
        'Test Loss': round(r['test_loss'], 4),
        'Train Time (min)': round(r['train_time_min'], 2),
        'Params': r['params'],
        'Checkpoint': r['ckpt_path'],
    })

ablation_df = pd.DataFrame(rows).sort_values('Test Top-1 (%)', ascending=False)

display(ablation_df)

ablation_path = os.path.join(PROCESSED_DIR, 'ablation_full250_four_models.csv')
ablation_df.to_csv(ablation_path, index=False)

print("Saved:", ablation_path)

,Model,Best Val Top-1 (%),Test Top-1 (%),Test Top-5 (%),Test Top-10 (%),Test Loss,Train Time (min),Params,Checkpoint
2,StrongTransformer,59.10,67.95,86.21,89.76,1.9149,154.77,6077690,/content/drive/MyDrive/UChicago/Masters/Spring...
3,StrongTCN,58.06,66.26,85.69,90.04,1.9563,44.99,1269370,/content/drive/MyDrive/UChicago/Masters/Spring...
0,StrongCNN,58.33,65.19,86.07,90.51,1.9732,43.69,1269370,/content/drive/MyDrive/UChicago/Masters/Spring...
1,StrongCNN_GRU,57.60,62.97,83.65,88.52,2.1159,133.82,5548666,/content/drive/MyDrive/UChicago/Masters/Spring...


Saved: /content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all250/ablation_full250_four_models.csv


In [28]:
# ============================================================
# CELL 27 — Prediction Helper
# ============================================================
@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    for x, y, lengths in loader:
        x = x.to(device)
        lengths = lengths.to(device)

        logits = model(x, lengths)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())

    return np.array(all_labels), np.array(all_preds)

In [29]:
# ============================================================
# CELL 28 — Best Model Classification Report
# ============================================================
best_result = max(results.values(), key=lambda r: r['test_top1'])
best_model_name = best_result['model']

print("Best model:", best_model_name)
print("Test Top-1:", best_result['test_top1'])
print("Test Top-5:", best_result['test_top5'])

best_model = model_builders[best_model_name](n_classes=N_CLASSES).to(device)
best_model.load_state_dict(torch.load(best_result['ckpt_path'], map_location=device))

y_true, y_pred = get_predictions(best_model, test_loader, device)

target_names = [idx_to_sign[i] for i in range(N_CLASSES)]

report = classification_report(
    y_true,
    y_pred,
    labels=list(range(N_CLASSES)),
    target_names=target_names,
    digits=3,
    zero_division=0
)

print(report)

report_path = os.path.join(PROCESSED_DIR, f'classification_report_full250_{best_model_name}.txt')

with open(report_path, 'w') as f:
    f.write(report)

print("Saved:", report_path)

Best model: StrongTransformer
Test Top-1: 0.6795228495392748
Test Top-5: 0.8621068545567092


/tmp/ipykernel_13064/3882962478.py:35: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)


              precision    recall  f1-score   support

          TV      0.864     0.704     0.776        54
       after      0.273     0.310     0.290        29
    airplane      0.979     0.839     0.904        56
         all      0.781     0.676     0.725        37
   alligator      0.814     0.686     0.745        51
      animal      0.484     0.652     0.556        46
     another      0.787     0.860     0.822        43
         any      0.677     0.750     0.712        28
       apple      0.765     0.830     0.796        47
         arm      0.605     0.821     0.697        28
        aunt      0.849     0.818     0.833        55
       awake      0.457     0.382     0.416        55
    backyard      0.533     0.545     0.539        44
         bad      0.507     0.826     0.628        46
     balloon      0.710     0.907     0.797        54
        bath      0.763     0.604     0.674        48
     because      0.900     0.900     0.900        60
         bed      0.846    

In [30]:
# ============================================================
# CELL 29 — Export Best Model to ONNX
# ============================================================
!pip install onnx onnxruntime onnxscript -q

import onnx
import onnxruntime as ort
import onnxscript

best_model = model_builders[best_model_name](n_classes=N_CLASSES).to(device)
best_model.load_state_dict(torch.load(best_result['ckpt_path'], map_location=device))
best_model.eval()

dummy_x = torch.randn(1, 96, 708).to(device)
dummy_l = torch.tensor([96]).to(device)

onnx_path = os.path.join(PROCESSED_DIR, f'sign_model_full250_{best_model_name}.onnx')

torch.onnx.export(
    best_model,
    (dummy_x, dummy_l),
    onnx_path,
    input_names=['x', 'lengths'],
    output_names=['logits'],
    dynamic_axes={
        'x': {0: 'batch_size'},
        'lengths': {0: 'batch_size'},
        'logits': {0: 'batch_size'},
    },
    opset_version=17
)

print("Saved ONNX:", onnx_path)

sess = ort.InferenceSession(onnx_path)

out = sess.run(None, {
    'x': dummy_x.cpu().numpy(),
    'lengths': dummy_l.cpu().numpy(),
})

print("ONNX output shape:", out[0].shape)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 14.9 MB/s eta 0:00:00


/tmp/ipykernel_13064/3882962478.py:35: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
/tmp/ipykernel_13064/1549477955.py:19: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0520 03:16:44.246000 13064 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX fe

[torch.onnx] Obtain model graph for `StrongSignTransformer([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `StrongSignTransformer([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/exporter/_onnx_program.py:460: UserWarning: # The axis name: batch_size will not be used, since it shares the same shape constraints with another axis: batch_size.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Saved ONNX: /content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all250/sign_model_full250_StrongTransformer.onnx
ONNX output shape: (1, 250)


In [31]:
# ============================================================
# CELL 30 — Save Final Summary
# ============================================================
summary = {
    'n_classes': N_CLASSES,
    'models_trained': MODELS_TO_TRAIN,
    'best_model': best_model_name,
    'best_checkpoint': best_result['ckpt_path'],
    'onnx_path': onnx_path,
    'test_top1': float(best_result['test_top1']),
    'test_top5': float(best_result['test_top5']),
    'test_top10': float(best_result['test_top10']),
    'label_map_path': label_map_path,
    'ablation_path': ablation_path,
    'classification_report_path': report_path,
}

summary_path = os.path.join(PROCESSED_DIR, 'full250_training_summary.json')

with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("Saved:", summary_path)

{
  "n_classes": 250,
  "models_trained": [
    "StrongCNN",
    "StrongCNN_GRU",
    "StrongTransformer",
    "StrongTCN"
  ],
  "best_model": "StrongTransformer",
  "best_checkpoint": "/content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all250/best_full250_StrongTransformer.pt",
  "onnx_path": "/content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all250/sign_model_full250_StrongTransformer.onnx",
  "test_top1": 0.6795228495392748,
  "test_top5": 0.8621068545567092,
  "test_top10": 0.8975772844847812,
  "label_map_path": "/content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all250/full250_label_map.json",
  "ablation_path": "/content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all250/ablation_full250_four_models.csv",
  "classification_report_path": "/content/drive/MyDrive/UChicago/Masters/Spring/ML2/SignBridge/data/raw/arthur/signbridge_all2